# BC–WA PM2.5 forecasting benchmark on Colab

Trains AirLapseV2 and baseline models on the BC + Washington hourly PM2.5
dataset that ships in this repo (`data/`), and scores 24-hour forecasts.

**Runtime → Change runtime type → GPU** before running. All metric files are
saved to Google Drive (`METRICS_DIR`), so a disconnected runtime loses no results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# metrics (benchmark.csv, per-run metric files, logs) -> Drive; the large
# prediction arrays and checkpoints stay on the runtime's local disk
METRICS_DIR = '/content/drive/MyDrive/bcwa_pm25_results'
LOCAL_RESULTS = '/content/results'

%cd /content
!rm -rf BC-WA-PM2.5-Forecast && git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git
%cd /content/BC-WA-PM2.5-Forecast
!pip install -q -r requirements.txt
import os, torch
os.environ['METRICS_DIR'] = METRICS_DIR
os.environ['RESULTS_DIR'] = LOCAL_RESULTS
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## Quick check (a few minutes)

One epoch of GRU, just to confirm everything runs. Its scores are meaningless.

In [ ]:
!METRICS_DIR="{METRICS_DIR}/smoke_test" python benchmark.py --models GRU --epochs 1 --repeats 1

## Benchmark

Each model is trained **5 times** (seeds 0-4) and the scores are reported as
**mean ± std** over the 5 runs. Every finished run is written to
`benchmark.csv` on Drive straight away; re-running the cell skips runs that
are already there, so after a disconnect it carries on from the run it
stopped in. `runs/<model>/` keeps train.py's own metric file per run and
`logs/` the full training output - all on Drive.

The 15 models: AirLapseV2 (proposed); MLP, LSTM, GRU; Transformer, Informer,
Autoformer, Crossformer; PM25_GNN, AirFormer, AirPhyNet, AirDualODE, AirDDE;
and two recent PM2.5 models re-implemented from their papers, TCN_DIR (Seo et
al. 2026) and STMamba (Zhang et al. 2025). 5 x 15 runs will take more than
one free Colab session - just re-run the cell when it stops.


In [ ]:
# Fast models first; the slow ones (AirDDE, AirFormer, TCN_DIR) last.
MODELS = ('MLP GRU LSTM PM25_GNN AirLapseV2 Crossformer AirPhyNet Transformer '
          'AirDualODE STMamba Informer Autoformer TCN_DIR AirFormer AirDDE')
!python benchmark.py --models {MODELS} --repeats 5 --epochs 50 --early_stop 10


In [ ]:
# mean ± std over the 5 runs per model (sample std), best RMSE first
import pandas as pd
s = pd.read_csv(f'{METRICS_DIR}/benchmark_summary.csv', index_col=0)
cols = ['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']
table = pd.DataFrame({'runs': s.n_repeats})
for c in cols:
    d = 1 if c == 'MAPE' else 3
    table[c] = [f'{m:.{d}f} ± {sd:.{d}f}' if pd.notna(sd) else f'{m:.{d}f}'
                for m, sd in zip(s[c + '_mean'], s[c + '_std'])]
table
